# Baselines

Edit **DATASET** and **ENCODER** in the code cell below, then run all.

In [1]:
# pip install -U nltk pandas scikit-learn sentence-transformers faiss-cpu

In [2]:
import sys
from pathlib import Path
import os
import time
import json
import hashlib

root = Path.cwd()
if not (root / "modules").is_dir():
    REPO_ROOT = root.parent
sys.path.insert(0, str(REPO_ROOT))

from utils.preprocessing import preprocess
from utils.utils import dataset_table_path,save_results
from modules.encoders import get_encoder
from modules.clustering import get_clusterer
from evaluation.evaluator import Evaluator
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import random

import numpy as np
import pandas as pd
import yaml

In [3]:
device = "cuda"

In [4]:
def estimate_k(embeddings, k_min=2, k_max=20, seed=42):
    scores = []
    for k in range(k_min, k_max + 1):
        
        model = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = model.fit_predict(embeddings)
        
        score = silhouette_score(embeddings, labels)
        scores.append((k, score))

        if score > best_score:
            best_score = score
            best_k = k

    return best_k, scores

In [5]:
def run_baseline(dataset_name, encoder_name, clusterer_name, cfg,device,max_samples=150000):
    cfg["dataset"]["max_samples"] =  max_samples
    cfg["dataset"]["name"] = dataset_name
    cfg["encoder"]["name"] = encoder_name
    cfg["clustering"]["name"] = clusterer_name

    path = dataset_table_path(REPO_ROOT, dataset_name)
    df = pd.read_csv(path)

    text_col = cfg["dataset"]["text_col"]
    label_col = cfg["dataset"]["label_col"]
    cap = cfg["dataset"].get("max_samples")

    
    df = df.dropna(subset=[text_col, label_col])
    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].str.len() > 0]

    if cap is not None and cap < len(df):
        df = df.sample(n=cap, random_state=cfg["experiment"]["seed"])

    texts = df[text_col].tolist()
    y_true = df[label_col].astype(int).to_numpy()

    seed = cfg["experiment"]["seed"]
    random.seed(seed)
    np.random.seed(seed)

    seed_warning = None
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception as e:
        seed_warning = str(e)
        print(f"[WARN] Torch seed setup failed: {seed_warning}")

    
    emb_hash_input = {
        "encoder": cfg["encoder"],
        "preprocessing": cfg["preprocessing"],
        "dataset": dataset_name,
        "cap": cap
    }

    emb_cfg_hash = hashlib.md5(
        json.dumps(emb_hash_input, sort_keys=True).encode()
    ).hexdigest()[:8]

    embeddings_path = Path(
        f"../artifacts/embeddings/emb_{dataset_name}_{encoder_name}_{emb_cfg_hash}.npy"
    )
    embeddings_path.parent.mkdir(parents=True, exist_ok=True)

    t0 = time.time()

    if not embeddings_path.exists():
        print("Generating embeddings")

        if encoder_name in cfg["preprocessing"]["enabled_for"]:
            cleaned, tokenized = preprocess(texts)
        else:
            cleaned, tokenized = texts, None

        encoder = get_encoder(cfg['encoder'])
        embeddings = encoder.fit_transform(device,cleaned,tokenized_texts=tokenized)

        np.save(embeddings_path, embeddings)
    else:
        print("Embeddings loaded (cached)")
        embeddings = np.load(embeddings_path)

    embedding_time = time.time() - t0
    
    reduction_cfg = cfg.get("reduction") or {}

    use_pca = reduction_cfg.get("use_pca", False)
    
    if use_pca and reduction_cfg.get("method") == "pca":
       
        embeddings = PCA(
            n_components=reduction_cfg.get("n_components", 0.95),
            random_state=seed
        ).fit_transform(embeddings)

    
    true_k = len(np.unique(y_true))
    k_cfg = cfg["clustering"].get("k", "true")

    if k_cfg == "true":
        k = true_k
    elif k_cfg == "auto":
        print("Estimating k automatically...")
        k = estimate_k(
            embeddings,
            k_min=2,
            k_max=cfg["clustering"].get("k_max", 20),
            seed=seed
        )
        print(f"Estimated k: {k}")
    else:
        k = int(k_cfg)

    n_runs = cfg["experiment"].get("n_runs",1)

    eval_mode = cfg["evaluation"].get("mode", "both")

    all_metrics = []
    all_times = []
    epoch_metrics_runs = []

    metrics_dir = Path("../artifacts/training_metrics")
    metrics_dir.mkdir(parents=True, exist_ok=True)

    for run in range(n_runs):
        run_seed = seed + run
        np.random.seed(run_seed)
        random.seed(run_seed)

        t1 = time.time()
        clusterer = get_clusterer(cfg["clustering"])
        y_pred = clusterer.fit_predict(embeddings, k, true_labels=y_true)
        clustering_time = time.time() - t1

        run_epoch_metrics = getattr(clusterer, "training_history", [])
        if run_epoch_metrics is None:
            run_epoch_metrics = []

        for entry in run_epoch_metrics:
            if isinstance(entry, dict):
                entry.setdefault("run", run)
        epoch_metrics_runs.append(run_epoch_metrics)

        unique_clusters = len(np.unique(y_pred))
        collapse_flag = unique_clusters < k

        metrics = {}

        if eval_mode in ["label", "both"]:
            evaluator = Evaluator(
                metrics=cfg["evaluation"]["metrics"]["label"])
            metrics.update(evaluator.evaluate(y_true, y_pred))

        if eval_mode in ["intrinsic", "both"]:
            from sklearn.metrics import silhouette_score, davies_bouldin_score
            try:
                metrics["silhouette"] = silhouette_score(embeddings, y_pred)
                metrics["davies_bouldin"] = davies_bouldin_score(
                    embeddings, y_pred)
                metrics["intrinsic_error"] = None
            except Exception as e:
                metrics["silhouette"] = None
                metrics["davies_bouldin"] = None
                metrics["intrinsic_error"] = str(e)
                print(f"[WARN] Intrinsic metric computation failed: {metrics['intrinsic_error']}")

        metrics["collapse"] = collapse_flag

        all_metrics.append(metrics)
        all_times.append(clustering_time)


    def aggregate(metric_list):
        keys = metric_list[0].keys()
        agg = {}
        for k in keys:
            values = [m[k] for m in metric_list if m[k] is not None]
            if len(values) > 0 and isinstance(values[0], (int, float, np.floating)):
                agg[f"{k}_mean"] = float(np.mean(values))
                agg[f"{k}_std"] = float(np.std(values))
            else:
                agg[k] = values[0] if values else None
        return agg

    final_metrics = aggregate(all_metrics)

    run_stamp_input = {
        "dataset": dataset_name,
        "encoder": encoder_name,
        "clusterer": clusterer_name,
        "seed": seed,
        "max_samples": cap,
        "n_runs": n_runs,
        "experiment": cfg["experiment"].get("name"),
    }
    run_stamp = hashlib.md5(
        json.dumps(run_stamp_input, sort_keys=True).encode()
    ).hexdigest()[:8]

    epoch_metrics_payload = {
        "experiment_name": cfg["experiment"]["name"],
        "dataset": dataset_name,
        "encoder": encoder_name,
        "clusterer": clusterer_name,
        "k_used": k,
        "k_true": true_k,
        "n_runs": n_runs,
        "run_stamp": run_stamp,
        "seed_warning": seed_warning,
        "metrics_by_run": epoch_metrics_runs,
    }

    metrics_file = metrics_dir / f"{dataset_name}_{encoder_name}_{clusterer_name}_{run_stamp}_epoch_metrics.json"
    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump(epoch_metrics_payload, f, indent=2)

    results = {
        # Experiment metadata
        "experiment_name": cfg["experiment"]["name"],
        "dataset": dataset_name,
        "encoder": encoder_name,
        "clusterer": clusterer_name,

        # Data
        "n_docs": len(texts),
        "k_used": k,
        "k_true": true_k,

        # Timing
        "embedding_time": embedding_time,
        "clustering_time_mean": float(np.mean(all_times)),
        "clustering_time_std": float(np.std(all_times)),
        "total_time": embedding_time + float(np.mean(all_times)),

        # Label metrics
        "acc_mean": final_metrics.get("acc_mean"),
        "acc_std": final_metrics.get("acc_std"),
        "nmi_mean": final_metrics.get("nmi_mean"),
        "nmi_std": final_metrics.get("nmi_std"),
        "ari_mean": final_metrics.get("ari_mean"),
        "ari_std": final_metrics.get("ari_std"),
        "purity_mean": final_metrics.get("purity_mean"),
        "purity_std": final_metrics.get("purity_std"),

        # Intrinsic metrics
        "silhouette_mean": final_metrics.get("silhouette_mean"),
        "silhouette_std": final_metrics.get("silhouette_std"),
        "davies_bouldin_mean": final_metrics.get("davies_bouldin_mean"),
        "davies_bouldin_std": final_metrics.get("davies_bouldin_std"),

        # Stability
        "collapse_mean": final_metrics.get("collapse_mean"),
        "epoch_metrics_file": str(metrics_file),
    }

    save_results(results)

    return results

In [6]:
with open(REPO_ROOT / "config" / "default.yaml") as f:
    cfg = yaml.safe_load(f)

encoder = 'sbert'
datasets = ['bbc_news','agnews','20newsgroups','reuters','dbpedia']

clusterers = ['sdcn']
for dataset in datasets:
    for clusterer in clusterers:
        print(f"\nRunning baseline for {dataset}-{encoder}-{clusterer}")
        run_baseline(dataset,encoder,clusterer,cfg,device)


Running baseline for bbc_news-sbert-sdcn
Embeddings loaded (cached)
[SDCN] Epoch 0 | Loss=0.2424 | KL=0.1030 | CE=0.4550 | RE=0.0029
[SDCN] Epoch 10 | Loss=0.2610 | KL=0.1107 | CE=0.4914 | RE=0.0029
[SDCN] Epoch 20 | Loss=0.2750 | KL=0.1167 | CE=0.5181 | RE=0.0029
[SDCN] Epoch 30 | Loss=0.2845 | KL=0.1210 | CE=0.5355 | RE=0.0029
[SDCN] Epoch 40 | Loss=0.2906 | KL=0.1239 | CE=0.5459 | RE=0.0029
[SDCN] Epoch 50 | Loss=0.2945 | KL=0.1260 | CE=0.5522 | RE=0.0029
[SDCN] Epoch 60 | Loss=0.2974 | KL=0.1276 | CE=0.5563 | RE=0.0029
[SDCN] Epoch 70 | Loss=0.2996 | KL=0.1290 | CE=0.5590 | RE=0.0029
[SDCN] Epoch 80 | Loss=0.3012 | KL=0.1302 | CE=0.5605 | RE=0.0029
[SDCN] Epoch 90 | Loss=0.3023 | KL=0.1312 | CE=0.5607 | RE=0.0029
[SDCN] Epoch 100 | Loss=0.3030 | KL=0.1322 | CE=0.5599 | RE=0.0029
[SDCN] Epoch 110 | Loss=0.3034 | KL=0.1331 | CE=0.5582 | RE=0.0029
[SDCN] Epoch 120 | Loss=0.3035 | KL=0.1339 | CE=0.5557 | RE=0.0029
[SDCN] Epoch 130 | Loss=0.3033 | KL=0.1347 | CE=0.5525 | RE=0.0029
[SDC